# 🤟 Wasel v4 Pro — Live Sign Language Translator

**Just 2 steps:**
1. Run Cell 1 (Install) → **Restart Runtime** → Run Cell 2
2. Cell 2 loads the pre-trained model and launches live!

> ⚡ **No training needed. Pre-trained model included!**

In [ ]:
#@title 📦 Cell 1: Install (Run ONCE then Restart Runtime)
#@markdown After this completes: **Runtime → Restart session** → then run Cell 2

!pip install -q "gradio>=5.0.0" "mediapipe==0.10.14" "scikit-learn>=1.3.0" "opencv-python-headless"

import mediapipe as mp
v = mp.__version__
ok = hasattr(mp, 'solutions')
print(f"\nmediapipe {v} — solutions API: {'✅ YES' if ok else '❌ NO'}")
if ok:
    print("\n🎉 Done! Now go to Runtime → Restart session → then run Cell 2")
else:
    print("\n⚠️ Go to Runtime → Restart session → then run Cell 2 (skip this cell!)")

In [ ]:
#@title 🎥 Cell 2: Launch Live Translator
#@markdown Loads pre-trained model from the repo and starts translating!

import os, cv2, pickle, logging
import numpy as np
import mediapipe as mp
import gradio as gr
from collections import deque

logging.basicConfig(level=logging.INFO)
print(f"mediapipe: {mp.__version__}")

# ─── Clone repo & load pre-trained model ───
REPO = "/content/wasel_repo"
if not os.path.exists(REPO):
    print("⏳ Cloning repository...")
    !git clone -q https://github.com/eltaweelactuary/Wasel_v4_Official_Release.git {REPO}
print("✅ Repository ready.")

PKL = os.path.join(REPO, "GCP_Source_Code/wasel_v4_data/models/psl_classifier.pkl")
with open(PKL, 'rb') as f:
    classifier, label_encoder = pickle.load(f)
print(f"✅ Model loaded: {classifier.n_features_in_} features, {len(label_encoder.classes_)} signs")
print(f"🏷️ Signs: {list(label_encoder.classes_)}")

# ─── MediaPipe Holistic ───
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils
mp_styles = mp.solutions.drawing_styles
holistic = mp_holistic.Holistic(
    static_image_mode=False, model_complexity=1,
    min_detection_confidence=0.5, min_tracking_confidence=0.5
)
print("✅ MediaPipe Holistic ready.")

N_FEAT = classifier.n_features_in_

def extract_features(results):
    features = []
    if results.pose_landmarks:
        for lm in results.pose_landmarks.landmark:
            features.extend([lm.x, lm.y, lm.z])
    else:
        features.extend([0.0] * 99)
    if results.left_hand_landmarks:
        for lm in results.left_hand_landmarks.landmark:
            features.extend([lm.x, lm.y, lm.z])
    else:
        features.extend([0.0] * 63)
    if results.right_hand_landmarks:
        for lm in results.right_hand_landmarks.landmark:
            features.extend([lm.x, lm.y, lm.z])
    else:
        features.extend([0.0] * 63)
    feat = np.array(features)
    if len(feat) > N_FEAT:
        feat = feat[:N_FEAT]
    elif len(feat) < N_FEAT:
        feat = np.pad(feat, (0, N_FEAT - len(feat)))
    return feat

buf = deque(maxlen=10)

def translate(image):
    if image is None:
        return image
    try:
        rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = holistic.process(rgb)
        out = image.copy()

        # Draw full skeleton (body + fingers)
        if results.pose_landmarks:
            mp_drawing.draw_landmarks(out, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS, mp_styles.get_default_pose_landmarks_style())
        if results.left_hand_landmarks:
            mp_drawing.draw_landmarks(out, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS, mp_styles.get_default_hand_landmarks_style())
        if results.right_hand_landmarks:
            mp_drawing.draw_landmarks(out, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS, mp_styles.get_default_hand_landmarks_style())

        # Predict sign
        feat = extract_features(results)
        buf.append(feat)
        text = "Waiting for signs..."
        if len(buf) >= 3:
            avg = np.mean(np.array(list(buf)), axis=0).reshape(1, -1)
            probs = classifier.predict_proba(avg)[0]
            idx = np.argmax(probs)
            conf = float(probs[idx]) * 100
            if conf > 30.0:
                text = f"{label_encoder.inverse_transform([idx])[0].upper()} ({conf:.1f}%)"

        cv2.putText(out, text, (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 0), 3)
        return out
    except Exception as e:
        return image

# ─── Launch ───
demo = gr.Interface(
    fn=translate,
    inputs=gr.Image(sources=["webcam"], streaming=True, label="📸 Camera"),
    outputs=gr.Image(label="🤟 Live Translation"),
    live=True,
    title="🤟 Wasel v4 Pro — Live Sign Language Translator",
    description="Show sign language gestures — AI translates in real-time with body + hand skeleton!",
)
print("\n🎉 Launching Live Translator...")
print("📋 Share the public URL with anyone!\n")
demo.launch(share=True, quiet=False)